In [1]:
import maboss
import scanpy as sc
import liana as li
from pathlib import Path
import os

cwd=Path.cwd()
if cwd.name == "notebooks":
    os.chdir(cwd.parent) 
os.getcwd()

'/Users/max/Documents/repositories/MSB_project'

## Loading part


## Remarks 

### loading resources

Here i load RNA data, important is that we need to:
- use proper resource for LIG/REC pairs tokens 
- select proper in aggregate functions (many samples does met criterias )

In [2]:
PATH_data = Path('data/multimodal')
rna = sc.read(PATH_data / "sma_rna.h5ad")
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)

# important mouse data - for token genes
mouse_resource = li.rs.select_resource(resource_name='mouseconsensus')

In [3]:
rna

AnnData object with n_obs × n_vars = 3036 × 16486
    obs: 'in_tissue', 'array_row', 'array_col', 'x', 'y', 'lesion', 'region', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    uns: 'lesion_colors', 'log1p', 'region_colors', 'spatial'
    obsm: 'spatial'
    layers: 'counts'

In [4]:
# Run rank_aggregate
li.mt.rank_aggregate(rna, 
                     groupby='region',
                     resource=mouse_resource,
                     expr_prop=0.1,
                     use_raw = False,           # we change default layer
                     layer='counts',            # we apply specific layer
                     verbose=True)

Using the `counts` layer!
/Users/max/.micromamba/envs/sc_env/lib/python3.12/site-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
Make sure that normalized counts are passed!


Using provided `resource`.


/Users/max/.micromamba/envs/sc_env/lib/python3.12/site-packages/liana/method/_pipe_utils/_pre.py:149: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
0.27 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 3036 samples and 1079 features


/Users/max/.micromamba/envs/sc_env/lib/python3.12/functools.py:912: UserWarning: zero-centering a sparse array/matrix densifies it.
/Users/max/.micromamba/envs/sc_env/lib/python3.12/site-packages/liana/method/sc/_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.
/Users/max/.micromamba/envs/sc_env/lib/python3.12/site-packages/liana/method/sc/_liana_pipe.py:293: FutureWarning: Use uns (e.g. `k in adata.uns` or `sorted(adata.uns)`) instead of AnnData.uns_keys, AnnData.uns_keys is deprecated and will be removed in the future.
/Users/max/.micromamba/envs/sc_env/lib/python3.12/site-packages/liana/method/sc/_liana_pipe.py:296: FutureWarning: Use uns (e.g. `k in adata.uns` or `sorted(adata.uns)`) instead of AnnData.uns_keys, AnnData.uns_keys is deprecated and will be removed in the future.


Assuming that counts were `natural` log-normalized!
Running CellPhoneDB


100%|██████████| 1000/1000 [00:01<00:00, 636.47it/s]


Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR


In [5]:
rna.uns.keys()

dict_keys(['lesion_colors', 'log1p', 'region_colors', 'spatial', 'liana_res'])

In [6]:
from MaBossSpatial import SpatialBooleanPipeline 

In [7]:
# Setting up pipeline 

## initialization
pipeline = SpatialBooleanPipeline(
    ## adata file, REMARK: it need to contain liana results
    spatial_data=rna, 
    ## value with liana results, you can find it in adata.uns.keys()
    liana_uns_key='liana_res' 
) 

## loading PyMyBoss model
pipeline.SetMaBossModel(
    mode='pretrained',
    model_name='Creative_name',
    bnd_path='data/maboss/CellFate/CellFateModel.bnd',
    cfg_path='data/maboss/CellFate/CellFateModel.cfg'
)

pipeline.SetSpatialSettings(bandwidth=30.0, cutoff=0.05, kernel="gaussian")

pipeline.SetTimeLags(strategy="experimental", custom_lags={"TNF": 10.0, "FAS": 15.0})

pipeline.SetSimulationSettings(max_time=45.0, delta_t=5.0, sample_count=1000)

Successfully loaded pretrained MaBoSS model 'Creative_name' with 28 nodes.


AttributeError: 'TimeLagEstimator' object has no attribute 'calculate_lags'

In [8]:
target_cells = rna.obs_names[:2].tolist()
output_path = "data/pymyboss/real_simulation_results.csv"

pipeline.RunPipeline(target_cell_ids=target_cells, output_csv_path=output_path)

AttributeError: 'SpatialEnvironment' object has no attribute 'filter_simulation_nodes'